# Assignment 02 — CNN architectures and controlled ablations

Run cells from top to bottom in **Google Colab with a T4 GPU**. This notebook writes its
own `src/` modules, saves every completed run under `artifacts/`, and leaves the
reserved CIFAR-10 test split untouched until the final selection cell. If a session
disconnects, download `artifacts/` before the session expires and restore it before
rerunning; existing checkpoints are the evidence of completed runs.

The four scratch architecture runs use 96×96 images; A0–A3 use 64×64; A4 is a
separate ImageNet transfer branch at 96×96. Training uses Adam, sparse categorical
cross-entropy from logits, batch 16, and a five-epoch maximum.


In [ ]:
from pathlib import Path
Path('src').mkdir(exist_ok=True)
Path('src/__init__.py').touch()
Path('artifacts').mkdir(exist_ok=True)
import tensorflow as tf
USE_MIXED_PRECISION = True  # Set False if GPU training becomes unstable.
if USE_MIXED_PRECISION and tf.config.list_physical_devices('GPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Precision:', tf.keras.mixed_precision.global_policy())


TensorFlow: 2.21.0
GPU: []
Precision: <DTypePolicy "float32">


In [ ]:
%%writefile src/models.py
"""Assignment 02 CNNs. All classifiers return 10 float32 logits."""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


def _logits(x, name="logits"):
    return layers.Dense(10, dtype="float32", name=name)(x)


def build_alexnet(input_shape=(96, 96, 3)):
    """Five historical conv stages; LRN and grouped convolutions are omitted."""
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for name, filters, kernel, stride, pool in [
        ("conv1", 96, 11, 4, True),
        ("conv2", 256, 5, 1, True),
        ("conv3", 384, 3, 1, False),
        ("conv4", 384, 3, 1, False),
        ("conv5", 256, 3, 1, True),
    ]:
        x = layers.Conv2D(filters, kernel, strides=stride, padding="same",
                          activation="relu", name=name)(x)
        if pool:
            x = layers.MaxPooling2D(3, strides=2, padding="same",
                                    name=f"{name}_pool")(x)
    x = layers.Flatten(name="flatten")(x)
    for n in (1, 2):
        x = layers.Dense(4096, activation="relu", name=f"fc{n}")(x)
        x = layers.Dropout(0.5, name=f"dropout{n}")(x)
    return keras.Model(inputs, _logits(x), name="AlexNet_96")


def build_vgg16(input_shape=(96, 96, 3), gap_head=False):
    """Manual VGG16; gap_head is for parameter analysis only, not benchmark training."""
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for block, (count, width) in enumerate(zip((2, 2, 3, 3, 3),
                                                (64, 128, 256, 512, 512)), 1):
        for i in range(1, count + 1):
            x = layers.Conv2D(width, 3, padding="same", activation="relu",
                              name=f"block{block}_conv{i}")(x)
        x = layers.MaxPooling2D(2, strides=2, name=f"block{block}_pool")(x)
    if gap_head:
        x = layers.GlobalAveragePooling2D(name="gap")(x)
    else:
        x = layers.Flatten(name="flatten")(x)
        x = layers.Dense(4096, activation="relu", name="fc1")(x)
        x = layers.Dropout(0.5, name="dropout1")(x)
        x = layers.Dense(4096, activation="relu", name="fc2")(x)
        x = layers.Dropout(0.5, name="dropout2")(x)
    return keras.Model(inputs, _logits(x),
                       name="VGG16_GAP_analytic" if gap_head else "VGG16_96")


def _inception(x, name, c1, c3r, c3, c5r, c5, pool_proj):
    b1 = layers.Conv2D(c1, 1, padding="same", activation="relu",
                       name=f"{name}_1x1")(x)
    b2 = layers.Conv2D(c3r, 1, padding="same", activation="relu",
                       name=f"{name}_3x3_reduce")(x)
    b2 = layers.Conv2D(c3, 3, padding="same", activation="relu",
                       name=f"{name}_3x3")(b2)
    b3 = layers.Conv2D(c5r, 1, padding="same", activation="relu",
                       name=f"{name}_5x5_reduce")(x)
    b3 = layers.Conv2D(c5, 5, padding="same", activation="relu",
                       name=f"{name}_5x5")(b3)
    b4 = layers.MaxPooling2D(3, strides=1, padding="same",
                             name=f"{name}_pool")(x)
    b4 = layers.Conv2D(pool_proj, 1, padding="same", activation="relu",
                       name=f"{name}_pool_proj")(b4)
    spatial = [tuple(b.shape[1:3]) for b in (b1, b2, b3, b4)]
    assert len(set(spatial)) == 1, f"{name}: branch shapes differ: {spatial}"
    return layers.Concatenate(axis=-1, name=f"{name}_concat")([b1, b2, b3, b4])


def build_googlenet(input_shape=(96, 96, 3)):
    """Inception-v1 stem and nine modules; auxiliary heads are omitted."""
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(64, 7, strides=2, padding="same", activation="relu",
                      name="stem_7x7")(inputs)
    x = layers.MaxPooling2D(3, strides=2, padding="same", name="stem_pool1")(x)
    x = layers.Conv2D(64, 1, padding="same", activation="relu",
                      name="stem_1x1")(x)
    x = layers.Conv2D(192, 3, padding="same", activation="relu",
                      name="stem_3x3")(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same", name="stem_pool2")(x)
    specs = [
        ("3a", 64, 96, 128, 16, 32, 32),
        ("3b", 128, 128, 192, 32, 96, 64),
        ("4a", 192, 96, 208, 16, 48, 64),
        ("4b", 160, 112, 224, 24, 64, 64),
        ("4c", 128, 128, 256, 24, 64, 64),
        ("4d", 112, 144, 288, 32, 64, 64),
        ("4e", 256, 160, 320, 32, 128, 128),
        ("5a", 256, 160, 320, 32, 128, 128),
        ("5b", 384, 192, 384, 48, 128, 128),
    ]
    for spec in specs:
        name = spec[0]
        if name in ("4a", "5a"):
            x = layers.MaxPooling2D(3, strides=2, padding="same",
                                    name=f"before_{name}_pool")(x)
        x = _inception(x, f"inception_{name}", *spec[1:])
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.4, name="dropout")(x)
    return keras.Model(inputs, _logits(x), name="GoogLeNet_96")


def _basic_block(x, filters, stride, name):
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False,
                      name=f"{name}_conv1")(x)
    y = layers.BatchNormalization(name=f"{name}_bn1")(y)
    y = layers.ReLU(name=f"{name}_relu1")(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                      name=f"{name}_conv2")(y)
    y = layers.BatchNormalization(name=f"{name}_bn2")(y)
    if stride != 1 or x.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False,
                                 name=f"{name}_projection")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_projection_bn")(shortcut)
    assert tuple(y.shape[1:]) == tuple(shortcut.shape[1:]), \
        f"{name}: Add shape mismatch: {y.shape} versus {shortcut.shape}"
    return layers.ReLU(name=f"{name}_out")(
        layers.Add(name=f"{name}_add")([y, shortcut]))


def build_resnet18(input_shape=(96, 96, 3)):
    inputs = keras.Input(shape=input_shape)
    x = layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False,
                      name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.ReLU(name="stem_relu")(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same", name="stem_pool")(x)
    for stage, filters in enumerate((64, 128, 256, 512), 1):
        for block in (1, 2):
            stride = 2 if stage > 1 and block == 1 else 1
            x = _basic_block(x, filters, stride, f"stage{stage}_block{block}")
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    return keras.Model(inputs, _logits(x), name="ResNet18_96")


def build_ablation(variant, input_shape=(64, 64, 3)):
    """A0, A1, A2, A3-P, A3-R share widths and a 10-logit head."""
    if variant not in ("A0", "A1", "A2", "A3-P", "A3-R"):
        raise ValueError(f"Unknown variant: {variant}")
    inputs = keras.Input(shape=input_shape)
    x = inputs
    use_bn = variant != "A0"
    deep = variant in ("A3-P", "A3-R")
    residual = variant == "A3-R"
    for stage, filters in enumerate((32, 64), 1):
        shortcut = x
        x = layers.Conv2D(filters, 3, padding="same", use_bias=not use_bn,
                          name=f"stage{stage}_conv1")(x)
        if use_bn:
            x = layers.BatchNormalization(name=f"stage{stage}_bn1")(x)
        x = layers.ReLU(name=f"stage{stage}_relu1")(x)
        if deep:
            x = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                              name=f"stage{stage}_conv2")(x)
            x = layers.BatchNormalization(name=f"stage{stage}_bn2")(x)
            if residual:
                shortcut = layers.Conv2D(filters, 1, use_bias=False,
                                         name=f"stage{stage}_projection")(shortcut)
                shortcut = layers.BatchNormalization(
                    name=f"stage{stage}_projection_bn")(shortcut)
                assert tuple(x.shape[1:]) == tuple(shortcut.shape[1:])
                x = layers.Add(name=f"stage{stage}_add")([x, shortcut])
            x = layers.ReLU(name=f"stage{stage}_relu2")(x)
        x = layers.MaxPooling2D(2, name=f"stage{stage}_pool")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    return keras.Model(inputs, _logits(x), name=variant.replace("-", "_"))


def build_transfer(input_shape=(96, 96, 3), fine_tune=False):
    """ResNet50 receives externally preprocessed 0-255-scale images."""
    inputs = keras.Input(shape=input_shape)
    backbone = keras.applications.ResNet50(include_top=False, weights="imagenet",
                                           input_shape=input_shape)
    backbone.trainable = bool(fine_tune)
    if fine_tune:
        for layer in backbone.layers[:-20]:
            layer.trainable = False
        for layer in backbone.layers[-20:]:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    return keras.Model(inputs, _logits(x),
                       name="A4_T" if fine_tune else "A4_F")


BUILDERS = {
    "AlexNet": build_alexnet,
    "VGG16": build_vgg16,
    "GoogLeNet": build_googlenet,
    "ResNet18": build_resnet18,
}


Writing src/models.py


In [ ]:
%%writefile src/data.py
"""Fixed CIFAR-10 split and on-demand image preparation."""

from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split


CIFAR10_GOOGLE_MIRROR = (
    "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
    "cifar-10-python.tar.gz"
)
CIFAR10_SHA256 = "6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce"


def load_cifar_and_split(output_dir="artifacts"):
    # Keras 3.14.1 downloads from the slow Toronto host. Prime its exact cache
    # entry from the Google-hosted copy used in newer Keras code, with the same
    # official SHA-256 integrity check, before calling the standard loader.
    keras.utils.get_file(
        fname="cifar-10-batches-py-target",
        origin=CIFAR10_GOOGLE_MIRROR,
        extract=True,
        file_hash=CIFAR10_SHA256,
    )
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    y_train = y_train.reshape(-1).astype("int64")
    y_test = y_test.reshape(-1).astype("int64")
    all_indices = np.arange(len(y_train))
    train_idx, remainder = train_test_split(
        all_indices, train_size=12000, random_state=42, stratify=y_train)
    val_idx, _ = train_test_split(
        remainder, train_size=3000, random_state=42,
        stratify=y_train[remainder])
    assert len(train_idx) == 12000 and len(val_idx) == 3000
    assert not set(train_idx).intersection(val_idx)
    assert len(x_test) == 10000
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    np.savez(output_dir / "split_indices.npz", train=train_idx, val=val_idx)
    return (x_train, y_train), (x_test, y_test), train_idx, val_idx


def make_dataset(images, labels, indices, size, batch_size, *, training=False,
                 augment=False, transfer=False, seed=42):
    """Resize per batch in tf.data; never keep resized copies in RAM."""
    if indices is not None:
        images, labels = images[indices], labels[indices]
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        ds = ds.shuffle(min(len(images), 12000), seed=seed,
                        reshuffle_each_iteration=True)
    ds = ds.enumerate()

    def prepare(index, sample):
        image, label = sample
        image = tf.image.resize(tf.cast(image, tf.float32), (size, size))
        if transfer:
            # ResNet50 preprocess_input requires original [0, 255] scale.
            image = keras.applications.resnet50.preprocess_input(image)
        else:
            image = image / 255.0
            if augment and training:
                random_seed = tf.stack([tf.cast(seed, tf.int32),
                                        tf.cast(index, tf.int32)])
                image = tf.image.stateless_random_flip_left_right(
                    image, seed=random_seed)
                padding = max(1, round(size * 0.08))
                image = tf.pad(image, [[padding, padding], [padding, padding],
                                       [0, 0]], mode="REFLECT")
                image = tf.image.stateless_random_crop(
                    image, [size, size, 3], seed=random_seed + [0, 1])
                image = tf.image.stateless_random_contrast(
                    image, 0.9, 1.1, seed=random_seed + [0, 2])
                image = tf.clip_by_value(image, 0.0, 1.0)
        return image, label

    ds = ds.map(prepare, num_parallel_calls=tf.data.AUTOTUNE,
                deterministic=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


Writing src/data.py


In [ ]:
%%writefile src/experiments.py
"""Training, measurements, checkpoints, and plots for Assignment 02."""

import csv
import json
import platform
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from src.data import make_dataset


RESULT_COLUMNS = [
    "run_id", "architecture_variant", "input_size", "seed", "trainable_params",
    "total_params", "epochs", "best_val_accuracy", "macro_f1",
    "final_train_accuracy", "generalization_gap", "training_minutes",
    "peak_gpu_memory_mb", "key_change_caveat", "checkpoint",
]


def vgg_parameter_comparison():
    """Analytic VGG16 parameter totals for a 96x96 input and 10 classes."""
    channels = [3, 64, 64, 128, 128, 256, 256, 256,
                512, 512, 512, 512, 512, 512]
    conv_base = sum((3 * 3 * cin + 1) * cout
                    for cin, cout in zip(channels[:-1], channels[1:]))
    canonical_head = ((3 * 3 * 512 + 1) * 4096
                      + (4096 + 1) * 4096 + (4096 + 1) * 10)
    gap_head = (512 + 1) * 10
    return {
        "convolutional_base_params": conv_base,
        "canonical_classifier_params": canonical_head,
        "gap_classifier_params": gap_head,
        "canonical_total_params": conv_base + canonical_head,
        "gap_total_params": conv_base + gap_head,
        "saved_parameters": canonical_head - gap_head,
    }


def record_environment(output_dir="artifacts"):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    try:
        gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True,
                                  timeout=15).stdout
    except (OSError, subprocess.TimeoutExpired):
        gpu_info = "nvidia-smi unavailable"
    try:
        commit_hash = subprocess.run(["git", "rev-parse", "HEAD"],
                                     capture_output=True, text=True,
                                     timeout=5).stdout.strip() or None
    except (OSError, subprocess.TimeoutExpired):
        commit_hash = None
    info = {
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "keras": keras.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "gpus": [d.name for d in tf.config.list_physical_devices("GPU")],
        "nvidia_smi": gpu_info,
        "code_commit_hash": commit_hash,
    }
    (output_dir / "environment.json").write_text(
        json.dumps(info, indent=2), encoding="utf-8")
    return info


def _manual_parameter_check(model):
    layer = next(layer for layer in model.layers
                 if isinstance(layer, keras.layers.Conv2D))
    kh, kw, cin, cout = [int(v) for v in layer.kernel.shape]
    manual = kh * kw * cin * cout + (cout if layer.use_bias else 0)
    assert manual == layer.count_params()
    return {"layer": layer.name, "formula": f"{kh}*{kw}*{cin}*{cout}"
            + (f"+{cout}" if layer.use_bias else ""),
            "manual": manual, "keras": layer.count_params()}


def _fallback_diagram(name, path):
    stages = {
        "AlexNet": ["96x96 RGB", "11x11 conv 96 + pool", "5x5 conv 256 + pool",
                    "3x3 conv 384", "3x3 conv 384", "3x3 conv 256 + pool",
                    "FC 4096", "FC 4096", "10 logits"],
        "VGG16": ["96x96 RGB", "2 x 3x3 conv 64 + pool",
                  "2 x 3x3 conv 128 + pool", "3 x 3x3 conv 256 + pool",
                  "3 x 3x3 conv 512 + pool", "3 x 3x3 conv 512 + pool",
                  "FC 4096", "FC 4096", "10 logits"],
        "GoogLeNet": ["96x96 RGB", "7x7 / 1x1 / 3x3 stem",
                      "Inception 3a, 3b", "Inception 4a-4e",
                      "Inception 5a, 5b", "GAP", "10 logits"],
        "ResNet18": ["96x96 RGB", "7x7 stem + pool",
                     "2 basic blocks, 64", "2 basic blocks, 128",
                     "2 basic blocks, 256", "2 basic blocks, 512",
                     "Each block: 3x3 -> 3x3 + shortcut", "GAP", "10 logits"],
    }[name]
    fig, ax = plt.subplots(figsize=(7, max(5, len(stages) * 0.75)))
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, len(stages) - 0.5)
    ax.axis("off")
    for i, label in enumerate(stages):
        y = len(stages) - i - 1
        ax.text(0.5, y, label, ha="center", va="center",
                bbox={"boxstyle": "round,pad=0.5", "fc": "#e8f0ff",
                      "ec": "#315a99"})
        if i:
            ax.annotate("", xy=(0.5, y + 0.27), xytext=(0.5, y + 0.73),
                        arrowprops={"arrowstyle": "->", "color": "#315a99"})
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def inspect_architecture(name, model, output_dir="artifacts"):
    output_dir = Path(output_dir) / "architectures"
    output_dir.mkdir(parents=True, exist_ok=True)
    lines = []
    model.summary(print_fn=lambda line, **kwargs: lines.append(line))
    (output_dir / f"{name}_summary.txt").write_text(
        "\n".join(lines), encoding="utf-8")
    input_shape = tuple(int(v) for v in model.input_shape[1:])
    logits = model(tf.zeros((2,) + input_shape), training=False)
    assert tuple(logits.shape) == (2, 10), (name, logits.shape)
    assert bool(tf.reduce_all(tf.math.is_finite(logits)))
    checked = _manual_parameter_check(model)
    (output_dir / f"{name}_check.json").write_text(
        json.dumps({"forward_shape": [2, 10], "parameter_check": checked},
                   indent=2), encoding="utf-8")
    try:
        keras.utils.plot_model(model, to_file=str(output_dir / f"{name}.png"),
                               show_shapes=True, show_layer_names=True,
                               expand_nested=False, dpi=80)
        diagram = str(output_dir / f"{name}.png")
    except Exception as exc:
        diagram = str(output_dir / f"{name}.png")
        _fallback_diagram(name, diagram)
        print(f"Graphviz diagram unavailable ({exc}); saved stage diagram instead")
    return {"forward_shape": (2, 10), "parameter_check": checked,
            "diagram": diagram}


def _memory_before():
    try:
        tf.config.experimental.reset_memory_stats("GPU:0")
    except Exception:
        pass


def _peak_memory_mb():
    try:
        return round(tf.config.experimental.get_memory_info("GPU:0")["peak"]
                     / 1024**2, 1)
    except Exception:
        return None


def _plot_history(history, run_id, output_dir):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, key, title in zip(axes, ("accuracy", "loss"),
                              ("Accuracy", "Loss")):
        ax.plot(history[key], label="train")
        ax.plot(history[f"val_{key}"], label="validation")
        ax.set(xlabel="Epoch", title=f"{run_id}: {title}")
        ax.legend()
    fig.tight_layout()
    path = output_dir / "figures" / f"{run_id}_curves.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160)
    plt.close(fig)
    return path


def _append_result(row, output_dir):
    path = output_dir / "results.csv"
    prior = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=RESULT_COLUMNS)
    prior = prior[prior.run_id != row["run_id"]]
    pd.concat([prior, pd.DataFrame([row])], ignore_index=True).to_csv(path,
                                                                      index=False)


def train_run(run_id, name, model, x_train, y_train, train_idx, val_idx,
              *, size, seed=42, epochs=5, batch_size=16, augment=False,
              transfer=False, learning_rate=3e-4, caveat="",
              output_dir="artifacts"):
    """Fit one run; select by validation accuracy and retain every measured result."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    keras.utils.set_random_seed(seed)
    train_ds = make_dataset(x_train, y_train, train_idx, size, batch_size,
                            training=True, augment=augment, transfer=transfer,
                            seed=seed)
    val_ds = make_dataset(x_train, y_train, val_idx, size, batch_size,
                          transfer=transfer)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")])
    checkpoint = output_dir / "checkpoints" / f"{run_id}.weights.h5"
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    callback = keras.callbacks.ModelCheckpoint(
        str(checkpoint), monitor="val_accuracy", mode="max",
        save_best_only=True, save_weights_only=True)
    _memory_before()
    start = time.perf_counter()
    fit = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                    callbacks=[callback], verbose=2)
    minutes = (time.perf_counter() - start) / 60
    history = {k: [float(v) for v in values]
               for k, values in fit.history.items()}
    history_path = output_dir / "histories" / f"{run_id}.json"
    history_path.parent.mkdir(parents=True, exist_ok=True)
    history_path.write_text(json.dumps(history, indent=2), encoding="utf-8")
    _plot_history(history, run_id, output_dir)
    model.load_weights(str(checkpoint))
    y_pred = np.argmax(model.predict(val_ds, verbose=0), axis=1)
    y_true = y_train[val_idx]
    best_val = max(history["val_accuracy"])
    assert abs(accuracy_score(y_true, y_pred) - best_val) < 0.005
    total = model.count_params()
    trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
    row = {
        "run_id": run_id, "architecture_variant": name,
        "input_size": f"{size}x{size}", "seed": seed,
        "trainable_params": trainable, "total_params": total,
        "epochs": len(history["loss"]),
        "best_val_accuracy": round(best_val, 6),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 6),
        "final_train_accuracy": round(history["accuracy"][-1], 6),
        "generalization_gap": round(history["accuracy"][-1]
                                    - history["val_accuracy"][-1], 6),
        "training_minutes": round(minutes, 3),
        "peak_gpu_memory_mb": _peak_memory_mb(),
        "key_change_caveat": caveat,
        "checkpoint": str(checkpoint),
    }
    _append_result(row, output_dir)
    return row


def log_failure(run_id, exc, batch_size, output_dir="artifacts"):
    path = Path(output_dir) / "failed_runs.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps({"run_id": run_id, "batch_size": batch_size,
                                 "error": repr(exc), "time": time.time()}) + "\n")


def plot_comparisons(results_path="artifacts/results.csv", output_dir="artifacts"):
    frame = pd.read_csv(results_path)
    output_dir = Path(output_dir) / "figures"
    output_dir.mkdir(parents=True, exist_ok=True)
    for title, subset, filename in [
        ("Four architectures", frame[frame.run_id.str.startswith("arch_")],
         "architecture_comparison.png"),
        ("Ablation and transfer", frame[frame.run_id.str.startswith("A")],
         "ablation_comparison.png"),
    ]:
        if subset.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(subset.run_id, subset.best_val_accuracy)
        ax.set(ylabel="Best validation accuracy", title=title, ylim=(0, 1))
        ax.tick_params(axis="x", rotation=45)
        fig.tight_layout()
        fig.savefig(output_dir / filename, dpi=160)
        plt.close(fig)

    architecture_runs = frame[frame.run_id.str.startswith("arch_")]
    if not architecture_runs.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for run_id in architecture_runs.run_id:
            history_path = output_dir.parent / "histories" / f"{run_id}.json"
            if not history_path.exists():
                continue
            history = json.loads(history_path.read_text(encoding="utf-8"))
            for ax, metric in zip(axes, ("accuracy", "loss")):
                ax.plot(history[metric], label=f"{run_id} train")
                ax.plot(history[f"val_{metric}"], linestyle="--",
                        label=f"{run_id} val")
        for ax, metric in zip(axes, ("accuracy", "loss")):
            ax.set(xlabel="Epoch", title=f"Four architectures: {metric}")
            ax.legend(fontsize=6)
        fig.tight_layout()
        fig.savefig(output_dir / "four_architecture_curves.png", dpi=160)
        plt.close(fig)


def final_test_once(model, checkpoint, x_test, y_test, *, size,
                    batch_size=16, transfer=False, output_dir="artifacts"):
    output_dir = Path(output_dir)
    marker = output_dir / "final_test_result.json"
    if marker.exists():
        raise RuntimeError("Final test has already been recorded; do not repeat it.")
    model.load_weights(str(checkpoint))
    test_ds = make_dataset(x_test, y_test, None, size, batch_size,
                           transfer=transfer)
    prediction = np.argmax(model.predict(test_ds, verbose=1), axis=1)
    classes = ["airplane", "automobile", "bird", "cat", "deer", "dog",
               "frog", "horse", "ship", "truck"]
    matrix = confusion_matrix(y_test, prediction)
    top_confusions = sorted(
        ({"true": classes[i], "predicted": classes[j], "count": int(matrix[i, j])}
         for i in range(10) for j in range(10) if i != j),
        key=lambda item: item["count"], reverse=True)[:5]
    result = {"checkpoint": str(checkpoint),
              "test_accuracy": float(accuracy_score(y_test, prediction)),
              "test_macro_f1": float(f1_score(y_test, prediction,
                                               average="macro")),
              "class_names": classes,
              "confusion_matrix": matrix.tolist(),
              "top_confusions": top_confusions}
    marker.write_text(json.dumps(result, indent=2), encoding="utf-8")
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(10), classes, rotation=45, ha="right")
    ax.set_yticks(range(10), classes)
    ax.set(xlabel="Predicted class", ylabel="True class", title="Final test confusion matrix")
    fig.tight_layout()
    fig.savefig(output_dir / "figures" / "final_confusion_matrix.png", dpi=160)
    plt.close(fig)
    return result


Writing src/experiments.py


## Step 1 — Data split and environment

One fixed stratified 12,000/3,000 split is saved. The 10,000-image original test
partition is loaded but not evaluated here. Resizing happens in the `tf.data`
pipeline rather than making permanent enlarged copies.


In [ ]:
import json
import numpy as np
from tensorflow import keras
from src.data import load_cifar_and_split, make_dataset
from src.models import BUILDERS, build_ablation, build_transfer, build_vgg16
from src.experiments import (record_environment, inspect_architecture, train_run,
                             log_failure, plot_comparisons, final_test_once,
                             vgg_parameter_comparison)
(x_train, y_train), (x_test, y_test), train_idx, val_idx = load_cifar_and_split()
environment = record_environment()
print('Split sizes:', len(train_idx), len(val_idx), len(x_test))
print('Train class counts:', np.bincount(y_train[train_idx]))
print('Validation class counts:', np.bincount(y_train[val_idx]))
print('Environment saved to artifacts/environment.json')


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Split sizes: 12000 3000 10000
Train class counts: [1200 1200 1200 1200 1200 1200 1200 1200 1200 1200]
Validation class counts: [300 300 300 300 300 300 300 300 300 300]
Environment saved to artifacts/environment.json


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import pandas as pd
import json
import shutil

uploaded = files.upload()
assert "final_test_restore.zip" in uploaded

with ZipFile("final_test_restore.zip") as archive:
    archive.extractall("artifacts")

results = pd.read_csv("artifacts/results.csv")
row = results.loc[results.run_id == "A4-T_s42"].iloc[0]

Path("artifacts/selected_winner.json").write_text(json.dumps({
    "run_id": "A4-T_s42",
    "validation_accuracy": float(row.best_val_accuracy),
    "checkpoint": row.checkpoint
}, indent=2))

model = build_transfer(fine_tune=True)
print(final_test_once(
    model, row.checkpoint, x_test, y_test,
    size=96, transfer=True
))

output = shutil.make_archive("assignment02_final_test", "zip", "artifacts")
files.download(output)

Saving final_test_restore.zip to final_test_restore.zip
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
625/625 ━━━━━━━━━━━━━━━━━━━━ 339s 539ms/step
{'checkpoint': 'artifacts/checkpoints/A4-T_s42.weights.h5', 'test_accuracy': 0.8497, 'test_macro_f1': 0.8499209657568135, 'class_names': ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'], 'confusion_matrix': [[865, 11, 32, 11, 14, 0, 5, 11, 38, 13], [19, 900, 0, 5, 0, 0, 3, 2, 11, 60], [23, 0, 840, 40, 35, 16, 34, 7, 4, 1], [11, 4, 38, 743, 43, 100, 33, 17, 6, 5], [7, 1, 40, 25, 830, 19, 35, 40, 2, 1], [2, 0, 13, 126, 28, 794, 9, 25, 3, 0], [12, 1, 30, 43, 19, 13, 875, 5, 2, 0], [8, 0, 16, 21, 70, 26, 1, 851, 3, 4], [53, 9, 5, 1, 4, 0, 3, 2, 910, 13], [17, 64, 0, 5, 0, 0, 1, 2, 22, 889]], 'top_confusions': [{'true': 'dog', 'predicted': 'cat', 'count': 126}, {'true': 'cat', 'predicted': 'dog', 'count': 100}, {'true': 'horse', 'predicted': 'deer', 'count': 70}, {'true': 'truck', 'predicted': 'autom

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 2 — Brief learning-rate sanity check

Use a small training/validation slice to check that the common 3e-4 Adam setting
produces finite loss before the main runs. These are diagnostics, not benchmark rows.


In [ ]:
sanity = {}
for lr in (1e-4, 3e-4):
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)
    model = build_ablation('A0')
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
    small_train = make_dataset(x_train, y_train, train_idx[:512], 64, 16,
                               training=True, seed=42)
    small_val = make_dataset(x_train, y_train, val_idx[:512], 64, 16)
    history = model.fit(small_train, validation_data=small_val,
                        epochs=1, verbose=2).history
    sanity[str(lr)] = {k: float(v[-1]) for k, v in history.items()}
    assert np.isfinite(sanity[str(lr)]['loss'])
Path('artifacts/lr_sanity.json').write_text(json.dumps(sanity, indent=2))
print(sanity)


32/32 - 10s - 311ms/step - accuracy: 0.0977 - loss: 2.3027 - val_accuracy: 0.1035 - val_loss: 2.3018
32/32 - 3s - 81ms/step - accuracy: 0.1133 - loss: 2.3032 - val_accuracy: 0.1016 - val_loss: 2.3013
{'0.0001': {'accuracy': 0.09765625, 'loss': 2.302670955657959, 'val_accuracy': 0.103515625, 'val_loss': 2.3017654418945312}, '0.0003': {'accuracy': 0.11328125, 'loss': 2.303194761276245, 'val_accuracy': 0.1015625, 'val_loss': 2.3013482093811035}}


## Step 3 — Build and verify four architectures

For each model, save `model.summary()`, a diagram where Graphviz is available,
a manual parameter calculation, and proof of a 2×96×96×3 to 2×10 logits pass.

**Adaptations:** AlexNet uses same-padded convolutions and pooling for 96×96,
with LRN and grouped convolutions omitted; VGG16 retains all 13 convolutions and
two 4096-unit FC layers; GoogLeNet retains its stem and nine Inception modules
but omits auxiliary heads; ResNet-18 retains its 2-2-2-2 basic-block stages.
All output heads have 10 logits rather than ImageNet classes.


In [ ]:
architecture_checks = {}
vgg_actual = None
for name, builder in BUILDERS.items():
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)
    model = builder()
    architecture_checks[name] = inspect_architecture(name, model)
    print(name, model.count_params(), architecture_checks[name])
    if name == 'VGG16':
        vgg_actual = model.count_params()
    del model
vgg_comparison = vgg_parameter_comparison()
assert vgg_actual == vgg_comparison['canonical_total_params']
print('VGG16 canonical versus GAP totals:', vgg_comparison)
Path('artifacts/vgg_head_analysis.json').write_text(
    json.dumps(vgg_comparison, indent=2))


AlexNet 30010762 {'forward_shape': (2, 10), 'parameter_check': {'layer': 'conv1', 'formula': '11*11*3*96+96', 'manual': 34944, 'keras': 34944}, 'diagram': 'artifacts/architectures/AlexNet.png'}


VGG16 50415434 {'forward_shape': (2, 10), 'parameter_check': {'layer': 'block1_conv1', 'formula': '3*3*3*64+64', 'manual': 1792, 'keras': 1792}, 'diagram': 'artifacts/architectures/VGG16.png'}


GoogLeNet 5983802 {'forward_shape': (2, 10), 'parameter_check': {'layer': 'stem_7x7', 'formula': '7*7*3*64+64', 'manual': 9472, 'keras': 9472}, 'diagram': 'artifacts/architectures/GoogLeNet.png'}


ResNet18 11191242 {'forward_shape': (2, 10), 'parameter_check': {'layer': 'stem_conv', 'formula': '7*7*3*64', 'manual': 9408, 'keras': 9408}, 'diagram': 'artifacts/architectures/ResNet18.png'}
VGG16 canonical versus GAP totals: {'convolutional_base_params': 14714688, 'canonical_classifier_params': 35700746, 'gap_classifier_params': 5130, 'canonical_total_params': 50415434, 'gap_total_params': 14719818, 'saved_parameters': 35695616}


221

## Step 4 — Four scratch architecture runs (96×96)

Each run uses the identical fixed split, initial learning rate 3e-4, batch 16,
five epochs, and best-validation-accuracy checkpoint rule. Run this cell once;
it may take considerable time. A failed run is logged instead of silently dropped.


In [ ]:
ARCH_EPOCHS = 5
for name, builder in BUILDERS.items():
    run_id = f'arch_{name}_s42'
    if Path(f'artifacts/checkpoints/{run_id}.weights.h5').exists() and        Path('artifacts/results.csv').exists() and        run_id in Path('artifacts/results.csv').read_text():
        print('Already complete:', run_id)
        continue
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)
    try:
        model = builder()
        row = train_run(run_id, name, model, x_train, y_train, train_idx, val_idx,
                        size=96, epochs=ARCH_EPOCHS, batch_size=16,
                        caveat='Scratch model; 96x96 upscaled CIFAR-10; one seed')
        print(row)
    except Exception as exc:
        log_failure(run_id, exc, 16)
        print('FAILED:', run_id, repr(exc))
    finally:
        if 'model' in locals():
            del model


Epoch 1/5
750/750 - 33s - 44ms/step - accuracy: 0.1877 - loss: 2.1355 - val_accuracy: 0.2890 - val_loss: 1.8981
Epoch 2/5
750/750 - 14s - 18ms/step - accuracy: 0.3372 - loss: 1.7571 - val_accuracy: 0.3467 - val_loss: 1.8267
Epoch 3/5
750/750 - 14s - 18ms/step - accuracy: 0.4154 - loss: 1.5663 - val_accuracy: 0.4417 - val_loss: 1.5017
Epoch 4/5
750/750 - 13s - 17ms/step - accuracy: 0.4682 - loss: 1.4348 - val_accuracy: 0.4407 - val_loss: 1.4828
Epoch 5/5
750/750 - 14s - 18ms/step - accuracy: 0.5148 - loss: 1.3275 - val_accuracy: 0.5090 - val_loss: 1.3411


/content/src/experiments.py:184: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([prior, pd.DataFrame([row])], ignore_index=True).to_csv(path,


{'run_id': 'arch_AlexNet_s42', 'architecture_variant': 'AlexNet', 'input_size': '96x96', 'seed': 42, 'trainable_params': 30010762, 'total_params': 30010762, 'epochs': 5, 'best_val_accuracy': 0.509, 'macro_f1': 0.510136, 'final_train_accuracy': 0.514833, 'generalization_gap': 0.005833, 'training_minutes': 1.437, 'peak_gpu_memory_mb': 1053.8, 'key_change_caveat': 'Scratch model; 96x96 upscaled CIFAR-10; one seed', 'checkpoint': 'artifacts/checkpoints/arch_AlexNet_s42.weights.h5'}
Epoch 1/5
750/750 - 104s - 139ms/step - accuracy: 0.0972 - loss: 2.3037 - val_accuracy: 0.1000 - val_loss: 2.3027
Epoch 2/5
750/750 - 54s - 73ms/step - accuracy: 0.0932 - loss: 2.3032 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 3/5
750/750 - 54s - 72ms/step - accuracy: 0.0975 - loss: 2.3030 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 4/5
750/750 - 54s - 72ms/step - accuracy: 0.0922 - loss: 2.3031 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 5/5
750/750 - 54s - 72ms/step - accuracy: 0.0941 - loss: 2.30

## Step 5 — Controlled A0–A3 ablations (64×64)

The first pass uses seed 42. Set `RUN_SECOND_SEED=True` to repeat A0, A1, A2,
A3-P, and A3-R with seed 123 if quota permits. A3-R has projection shortcuts;
their extra parameters must be reported as a caveat against A3-P.


In [ ]:
RUN_SECOND_SEED = False
ABLATION_EPOCHS = 5
for seed in ((42, 123) if RUN_SECOND_SEED else (42,)):
    for variant in ('A0', 'A1', 'A2', 'A3-P', 'A3-R'):
        run_id = f'{variant}_s{seed}'
        if Path(f'artifacts/checkpoints/{run_id}.weights.h5').exists() and            Path('artifacts/results.csv').exists() and            run_id in Path('artifacts/results.csv').read_text():
            print('Already complete:', run_id)
            continue
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        try:
            model = build_ablation(variant)
            caveat = 'Projection shortcuts add a small parameter difference'                 if variant == 'A3-R' else 'Controlled 64x64 scratch ablation'
            row = train_run(run_id, variant, model, x_train, y_train,
                            train_idx, val_idx, size=64, seed=seed,
                            epochs=ABLATION_EPOCHS, batch_size=16,
                            augment=variant in ('A2', 'A3-P', 'A3-R'),
                            caveat=caveat)
            print(row)
        except Exception as exc:
            log_failure(run_id, exc, 16)
            print('FAILED:', run_id, repr(exc))
        finally:
            if 'model' in locals():
                del model


Epoch 1/5
750/750 - 8s - 11ms/step - accuracy: 0.1813 - loss: 2.1728 - val_accuracy: 0.1883 - val_loss: 2.1166
Epoch 2/5
750/750 - 4s - 6ms/step - accuracy: 0.2282 - loss: 2.0655 - val_accuracy: 0.2233 - val_loss: 2.0554
Epoch 3/5
750/750 - 4s - 5ms/step - accuracy: 0.2466 - loss: 2.0254 - val_accuracy: 0.2590 - val_loss: 2.0025
Epoch 4/5
750/750 - 4s - 5ms/step - accuracy: 0.2592 - loss: 1.9976 - val_accuracy: 0.2797 - val_loss: 1.9807
Epoch 5/5
750/750 - 4s - 6ms/step - accuracy: 0.2729 - loss: 1.9719 - val_accuracy: 0.2990 - val_loss: 1.9490
{'run_id': 'A0_s42', 'architecture_variant': 'A0', 'input_size': '64x64', 'seed': 42, 'trainable_params': 20042, 'total_params': 20042, 'epochs': 5, 'best_val_accuracy': 0.299, 'macro_f1': 0.27728, 'final_train_accuracy': 0.272917, 'generalization_gap': -0.026083, 'training_minutes': 0.401, 'peak_gpu_memory_mb': 1474.2, 'key_change_caveat': 'Controlled 64x64 scratch ablation', 'checkpoint': 'artifacts/checkpoints/A0_s42.weights.h5'}
Epoch 1/5
75

## Step 6 — Separate A4 transfer branch

A4-F uses an ImageNet-pretrained frozen ResNet50 with original 0–255 scale
passed through `resnet50.preprocess_input`. A4-T loads the A4-F checkpoint,
unfreezes selected high layers while keeping BatchNorm in inference mode,
recompiles at 1e-5, and trains again. These are not one-factor extensions of A3.


In [ ]:
for variant, fine_tune, lr in [('A4-F', False, 3e-4),
                                     ('A4-T', True, 1e-5)]:
    run_id = f'{variant}_s42'
    frozen_checkpoint = Path('artifacts/checkpoints/A4-F_s42.weights.h5')
    if fine_tune and not frozen_checkpoint.exists():
        print('Skipping A4-T: A4-F checkpoint is missing')
        break
    if Path(f'artifacts/checkpoints/{run_id}.weights.h5').exists() and        Path('artifacts/results.csv').exists() and        run_id in Path('artifacts/results.csv').read_text():
        print('Already complete:', run_id)
        continue
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)
    try:
        model = build_transfer(fine_tune=fine_tune)
        if fine_tune:
            model.load_weights(str(frozen_checkpoint))
        row = train_run(run_id, variant, model, x_train, y_train,
                        train_idx, val_idx, size=96, epochs=5, batch_size=16,
                        transfer=True, learning_rate=lr,
                        caveat='Separate ImageNet-pretrained transfer branch')
        print(row)
    except Exception as exc:
        log_failure(run_id, exc, 16)
        print('FAILED:', run_id, repr(exc))
    finally:
        if 'model' in locals():
            del model


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/5
750/750 - 52s - 70ms/step - accuracy: 0.7191 - loss: 0.8852 - val_accuracy: 0.8070 - val_loss: 0.5758
Epoch 2/5
750/750 - 55s - 73ms/step - accuracy: 0.8439 - loss: 0.4726 - val_accuracy: 0.8233 - val_loss: 0.5425
Epoch 3/5
750/750 - 20s - 27ms/step - accuracy: 0.8734 - loss: 0.3773 - val_accuracy: 0.8340 - val_loss: 0.5028
Epoch 4/5
750/750 - 20s - 27ms/step - accuracy: 0.8947 - loss: 0.3186 - val_accuracy: 0.8367 - val_loss: 0.5139
Epoch 5/5
750/750 - 16s - 21ms/step - accuracy: 0.9095 - loss: 0.2716 - val_accuracy: 0.8370 - val_loss: 0.5136
{'run_id': 'A4-F_s42', 'architecture_variant': 'A4-F', 'input_size': '96x96', 'seed': 42, 'trainable_params': 20490, 'total_params': 23608202, 'epochs': 5, 'best_val_accuracy': 0.837, 'macro_f1': 0.836677, 'final_train_accuracy': 0.9095, 'generalization_gap': 0.0725, 'training_minutes': 2.717, 'peak_gpu_memory_mb': 1598.7, 'key_change_caveat': 'Separate ImageNet-pretrained transfer bran

## Step 7 — Review validation evidence and choose final model

Generate the comparison figures, inspect `artifacts/results.csv`, and record
the winning run ID **before** the reserved test split is evaluated. Do not use
test performance to pick a model.


In [ ]:
import pandas as pd
results = pd.read_csv('artifacts/results.csv')
plot_comparisons()
display(results.sort_values('best_val_accuracy', ascending=False))
print('Figures are in artifacts/figures/')


,run_id,architecture_variant,input_size,seed,trainable_params,total_params,epochs,best_val_accuracy,macro_f1,final_train_accuracy,generalization_gap,training_minutes,peak_gpu_memory_mb,key_change_caveat,checkpoint
10,A4-T_s42,A4-T,96x96,42,8939530,23608202,5,0.860333,0.860470,0.999917,0.139583,2.557,1763.0,Separate ImageNet-pretrained transfer branch,artifacts/checkpoints/A4-T_s42.weights.h5
9,A4-F_s42,A4-F,96x96,42,20490,23608202,5,0.837000,0.836677,0.909500,0.072500,2.717,1598.7,Separate ImageNet-pretrained transfer branch,artifacts/checkpoints/A4-F_s42.weights.h5
3,arch_ResNet18_s42,ResNet18,96x96,42,11181642,11191242,5,0.521000,0.536160,0.771917,0.304917,2.189,3627.1,Scratch model; 96x96 upscaled CIFAR-10; one seed,artifacts/checkpoints/arch_ResNet18_s42.weight...
0,arch_AlexNet_s42,AlexNet,96x96,42,30010762,30010762,5,0.509000,0.510136,0.514833,0.005833,1.437,1053.8,Scratch model; 96x96 upscaled CIFAR-10; one seed,artifacts/checkpoints/arch_AlexNet_s42.weights.h5
7,A3-P_s42,A3-P,64x64,42,66410,66794,5,0.494333,0.472409,0.514083,0.019750,0.735,1629.0,Controlled 64x64 scratch ablation,artifacts/checkpoints/A3-P_s42.weights.h5
8,A3-R_s42,A3-R,64x64,42,68746,69322,5,0.476000,0.461140,0.497667,0.021667,0.825,1547.3,Projection shortcuts add a small parameter dif...,artifacts/checkpoints/A3-R_s42.weights.h5
6,A2_s42,A2,64x64,42,20138,20330,5,0.376333,0.366868,0.410083,0.054083,0.552,1418.2,Controlled 64x64 scratch ablation,artifacts/checkpoints/A2_s42.weights.h5
5,A1_s42,A1,64x64,42,20138,20330,5,0.372000,0.357546,0.421167,0.094833,0.456,1553.0,Controlled 64x64 scratch ablation,artifacts/checkpoints/A1_s42.weights.h5
4,A0_s42,A0,64x64,42,20042,20042,5,0.299000,0.277280,0.272917,-0.026083,0.401,1474.2,Controlled 64x64 scratch ablation,artifacts/checkpoints/A0_s42.weights.h5
2,arch_GoogLeNet_s42,GoogLeNet,96x96,42,5983802,5983802,5,0.100000,0.018182,0.096083,-0.003917,4.741,1585.3,Scratch model; 96x96 upscaled CIFAR-10; one seed,artifacts/checkpoints/arch_GoogLeNet_s42.weigh...


Figures are in artifacts/figures/


## Step 8 — Final test, exactly once

After reviewing validation results, set `WINNER_RUN_ID` to one run ID from the
table and execute this cell once. It records the choice before test prediction
and refuses to repeat if a final result already exists.


In [ ]:
WINNER_RUN_ID = ''  # e.g. 'arch_ResNet18_s42'; fill after reviewing validation results
if not WINNER_RUN_ID:
    print('Set WINNER_RUN_ID after reviewing validation results; test not run.')
else:
    selected = results.loc[results.run_id == WINNER_RUN_ID]
    assert len(selected) == 1, 'WINNER_RUN_ID must match exactly one result row'
    row = selected.iloc[0]
    selection_path = Path('artifacts/selected_winner.json')
    if not selection_path.exists():
        selection_path.write_text(json.dumps({
            'run_id': WINNER_RUN_ID,
            'validation_accuracy': float(row.best_val_accuracy),
            'checkpoint': row.checkpoint}, indent=2))
    else:
        assert json.loads(selection_path.read_text())['run_id'] == WINNER_RUN_ID
    keras.backend.clear_session()
    name = row.architecture_variant
    if name in BUILDERS:
        model = BUILDERS[name]()
    elif name in ('A4-F', 'A4-T'):
        model = build_transfer(fine_tune=(name == 'A4-T'))
    else:
        model = build_ablation(name)
    size = int(str(row.input_size).split('x')[0])
    test_result = final_test_once(model, row.checkpoint, x_test, y_test,
                                  size=size, transfer=name in ('A4-F', 'A4-T'))
    print(test_result)


Set WINNER_RUN_ID after reviewing validation results; test not run.


## Step 9 — Save submission evidence

Download the `artifacts/` directory before ending the Colab session. Include
`results.csv`, figures, summaries, logs, environment, split indices, notebook,
and the report in the GitHub repository. Never replace missing results with
numbers from papers or other experiments.


In [ ]:
import shutil
archive = shutil.make_archive('assignment02_artifacts', 'zip', 'artifacts')
print('Download:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass  # In local Jupyter, the zip remains in the current folder.


Download: /content/assignment02_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import shutil
import zipfile
root = Path('/content/artifacts')
found = [p for p in root.rglob('*') if p.is_file()]
print('Artifact files:', len(found))
print('Runtime files:', [p.name for p in Path('/content').iterdir()])
if found:
    archive = shutil.make_archive('/content/assignment02_artifacts', 'zip', root)
    with zipfile.ZipFile(archive) as saved_zip:
        print('Files in ZIP:', len(saved_zip.namelist()))
    from google.colab import files
    files.download(archive)
else:
    print('No files to download. The previous runtime files are gone; old notebook outputs can still be visible.')

Artifact files: 53
Runtime files: ['.config', 'artifacts', 'assignment02_artifacts.zip', 'src', 'sample_data']
Files in ZIP: 57


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>